In [29]:
import torch
import os 

def random_rotation_matrix(max_angle_deg=25, device='cpu'):
    max_angle_rad = max_angle_deg * torch.pi / 180
    
    # 회전 축: uniform(-1,1)로 생성 후 단위 벡터로 정규화
    axis = torch.empty(3, device=device).uniform_(-1, 1)
    axis = axis / axis.norm()
    
    # 회전 각도: uniform(0, max_angle_rad)
    angle = torch.empty(1, device=device).uniform_(0, max_angle_rad)
    
    # Rodrigues' rotation formula
    K = torch.tensor([[0,-axis[2],axis[1]],
                      [axis[2],0,-axis[0]],
                      [-axis[1],axis[0],0]], device=device)
    R = torch.eye(3, device=device) + torch.sin(angle)*K + (1-torch.cos(angle))*(K@K)
    return R

def random_translation(max_shift=2.0, device='cpu'):
    # uniform(-max_shift, max_shift)
    t = torch.empty(3, device=device).uniform_(-max_shift, max_shift)
    return t

def transform_antibody(pdb_input, pdb_output, max_angle_deg=20, max_shift=0.0, device='cpu'):
    with open(pdb_input) as f:
        lines = f.readlines()

    # PDB에서 실제 등장하는 첫 두 체인을 찾기
    chains = []
    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            chain = line[21]
            if chain not in chains:
                chains.append(chain)
            if len(chains) >= 2:
                break
    antibody_chains = chains[:2]
    print(f"First two chains (antibody) will be transformed: {antibody_chains}")

    # 랜덤 rotation과 translation 생성
    R = random_rotation_matrix(max_angle_deg, device=device)
    t = random_translation(max_shift, device=device)

    new_lines = []
    for line in lines:
        if line.startswith("ATOM") or line.startswith("HETATM"):
            chain = line[21]
            if chain in antibody_chains:
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coord = torch.tensor([x, y, z], device=device)
                new_coord = (R @ coord + t).tolist()
                line = line[:30] + f"{new_coord[0]:8.3f}{new_coord[1]:8.3f}{new_coord[2]:8.3f}" + line[54:]
        new_lines.append(line)

    # 파일 저장
    with open(pdb_output, 'w') as f:
        f.writelines(new_lines)
    print(f"Saved transformed PDB to {pdb_output}")


# 사용 예시
src_pdb = '/home/psh/protein-frame-flow/inference_outputs/CDRFlow_v1.2.2_acc2/2025-07-10_16-06-11/epoch=133-step=192156_copy/run_2025-07-15_09-11-34/7df1_F_J_C/sample_0/sample_1.pdb'
max_angle = 20
max_shift = 5.0
if not os.path.exists(f'/home/psh/protein-frame-flow/notebook/angle_{max_angle}_shift_{max_shift}'):
    os.makedirs(f'/home/psh/protein-frame-flow/notebook/angle_{max_angle}_shift_{max_shift}')
for i in range(50):
    target_pdb = f'/home/psh/protein-frame-flow/notebook/angle_{max_angle}_shift_{max_shift}/7df1_{i}.pdb'
    transform_antibody(src_pdb, target_pdb, max_angle_deg=20, max_shift=5.0, device='cpu')


First two chains (antibody) will be transformed: ['f', 'j']
Saved transformed PDB to /home/psh/protein-frame-flow/notebook/angle_20_shift_5.0/7df1_0.pdb
First two chains (antibody) will be transformed: ['f', 'j']
Saved transformed PDB to /home/psh/protein-frame-flow/notebook/angle_20_shift_5.0/7df1_1.pdb
First two chains (antibody) will be transformed: ['f', 'j']
Saved transformed PDB to /home/psh/protein-frame-flow/notebook/angle_20_shift_5.0/7df1_2.pdb
First two chains (antibody) will be transformed: ['f', 'j']
Saved transformed PDB to /home/psh/protein-frame-flow/notebook/angle_20_shift_5.0/7df1_3.pdb
First two chains (antibody) will be transformed: ['f', 'j']
Saved transformed PDB to /home/psh/protein-frame-flow/notebook/angle_20_shift_5.0/7df1_4.pdb
First two chains (antibody) will be transformed: ['f', 'j']
Saved transformed PDB to /home/psh/protein-frame-flow/notebook/angle_20_shift_5.0/7df1_5.pdb
First two chains (antibody) will be transformed: ['f', 'j']
Saved transformed PDB 

In [30]:
import torch

def random_rotation_matrices(batch_size, max_angle_deg=20, device='cpu'):
    """
    batch_size 개수만큼 랜덤 회전 행렬 생성
    output: (B, 3, 3)
    """
    max_angle_rad = max_angle_deg * torch.pi / 180

    # 회전축: uniform(-1,1), 정규화
    axis = torch.empty(batch_size, 3, device=device).uniform_(-1, 1)
    axis = axis / axis.norm(dim=-1, keepdim=True)

    # 회전 각도: uniform(0, max_angle_rad)
    angle = torch.empty(batch_size, 1, device=device).uniform_(0, max_angle_rad)

    # Rodrigues' formula batch 적용
    K = torch.zeros(batch_size, 3, 3, device=device)
    K[:, 0, 1] = -axis[:, 2]
    K[:, 0, 2] =  axis[:, 1]
    K[:, 1, 0] =  axis[:, 2]
    K[:, 1, 2] = -axis[:, 0]
    K[:, 2, 0] = -axis[:, 1]
    K[:, 2, 1] =  axis[:, 0]

    I = torch.eye(3, device=device).unsqueeze(0).expand(batch_size, -1, -1)
    sinA = torch.sin(angle)[:, None]
    cosA = torch.cos(angle)[:, None]

    R = I + sinA * K + (1 - cosA) * (K @ K)
    return R  # (B, 3, 3)


def random_translations(batch_size, max_shift=5.0, device='cpu'):
    """
    batch_size 개수만큼 랜덤 translation 생성
    output: (B, 3)
    """
    t = torch.empty(batch_size, 3, device=device).uniform_(-max_shift, max_shift)
    return t  # (B, 3)

In [33]:
B = 2
rot = random_rotation_matrices(B, max_angle_deg=20)
trans = random_translations(B, max_shift=5.0)

print(rot)   # torch.Size([4, 3, 3])
print(trans) # torch.Size([4, 3])

tensor([[[ 0.9662, -0.2114, -0.1479],
         [ 0.1737,  0.9569, -0.2329],
         [ 0.1907,  0.1993,  0.9612]],

        [[ 0.9900, -0.0432,  0.1344],
         [ 0.0669,  0.9820, -0.1767],
         [-0.1243,  0.1839,  0.9750]]])
tensor([[ 0.7621,  0.5236,  3.4851],
        [-3.6209, -3.2674, -0.3666]])
